# IMPORTACIONES

In [32]:
from google.colab import drive
import pandas as pd
import numpy as np
import logging
import ast
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

# CONEXIÓN DE DRIVE

In [ ]:
# Montamos Drive en nuestro espacio de trabajo
drive.mount('/content/drive')

# Visualizamos las primeras filas para verificar alérgenos e ingredientes
print(data.head())

Mounted at /content/drive
   Unnamed: 0                                              Title  \
0           0  Miso-Butter Roast Chicken With Acorn Squash Pa...   
1           1                    Crispy Salt and Pepper Potatoes   
2           2                        Thanksgiving Mac and Cheese   
3           3                 Italian Sausage and Bread Stuffing   
4           4                                       Newton's Law   

                                         Ingredients  \
0  ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...   
1  ['2 large egg whites', '1 pound new potatoes (...   
2  ['1 cup evaporated milk', '1 cup whole milk', ...   
3  ['1 (¾- to 1-pound) round Italian loaf, cut in...   
4  ['1 teaspoon dark brown sugar', '1 teaspoon ho...   

                                        Instructions  \
0  Pat chicken dry with paper towels, season all ...   
1  Preheat oven to 400°F and line a rimmed baking...   
2  Place a rack in middle of oven; preheat to 400...   
3  P

# Modelo KNN Con 13K RECIPES

In [ ]:


# =========================
# 1. PREPARAR DATOS
# =========================
data = pd.read_csv('/content/drive/MyDrive/DIET-IA/dataset/13k-recipes.csv')
df = data.copy()
df = df.dropna(subset=['Title', 'Cleaned_Ingredients', 'Instructions'])

def join_ingredients(ing_str):
    try:
        lista = ast.literal_eval(ing_str)
        return " ".join(lista)
    except:
        return str(ing_str)

df['Features'] = df['Cleaned_Ingredients'].apply(join_ingredients)

# Filtrar recetas muy raras (estabilidad)
df = df[df['Title'].map(df['Title'].value_counts()) > 1]

# =========================
# 2. CREAR EMBEDDINGS
# =========================
print("Generando embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    df['Features'].tolist(),
    show_progress_bar=True
)

# =========================
# 3. ENTRENAR BUSCADOR VECINAL
# =========================
nn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

nn.fit(embeddings)

# =========================
# 4. FUNCIÓN FINAL TOP-5
# =========================
def diet_ia_recommender(ingredientes_usuario, top_k=5):
    emb = embedder.encode([ingredientes_usuario])
    distances, indices = nn.kneighbors(emb, n_neighbors=top_k)

    resultados = []

    for rank, idx in enumerate(indices[0]):
        receta = df.iloc[idx]

        resultados.append({
            "Rank": rank + 1,
            "Similitud": float(1 - distances[0][rank]),
            "Receta": receta['Title'],
            "Ingredientes": receta['Ingredients'],
            "Instrucciones": receta['Instructions']
        })

    return resultados

Generando embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [6]:
diet_ia_recommender("rice, chicken, potato, garlic")

[{'Rank': 1,
  'Similitud': 0.6387215852737427,
  'Receta': 'Chicken Tikka Masala',
  'Ingredientes': "['6 garlic cloves, finely grated', '4 teaspoons finely grated peeled ginger', '4 teaspoons ground turmeric', '2 teaspoons garam masala', '2 teaspoons ground coriander', '2 teaspoons ground cumin', '1 1/2 cups whole-milk yogurt (not Greek)', '1 tablespoon kosher salt', '2 pounds skinless, boneless chicken breasts, halved lengthwise', '3 tablespoons ghee (clarified butter) or vegetable oil', '1 small onion, thinly sliced', '1/4 cup tomato paste', '6 cardamom pods, crushed', '2 dried chiles de árbol or 1/2 teaspoon crushed red pepper flakes', '1 28-ounce can whole peeled tomatoes', '2 cups heavy cream', '3/4 cup chopped fresh cilantro plus sprigs for garnish', 'Steamed basmati rice (for serving)']",
  'Instrucciones': 'Combine garlic, ginger, turmeric, garam masala, coriander, and cumin in a small bowl. Whisk yogurt, salt, and half of spice mixture in a medium bowl; add chicken and turn 

# MODELO KNN CON RECIPES 20K

In [30]:
# =========================
# 1. PREPARAR DATOS
# =========================
data = pd.read_csv('/content/drive/MyDrive/DIET-IA/dataset/recetas.csv')
df = data.copy()

# Renombrar columnas
df = df.rename(columns={
    "title": "Title",
    "ingredients": "Ingredients",
    "directions": "Instructions"
})

# 1. Eliminar nulos
df = df.dropna(subset=["Title", "Ingredients", "Instructions"])

# 2. Eliminar duplicados (Mantiene solo una copia de cada receta)
df = df.drop_duplicates(subset=["Title"]).reset_index(drop=True)

# 3. Limpiar ingredientes
def join_ingredients(ing_str):
    if isinstance(ing_str, str):
        # Reemplaza el separador "|" por espacios para que el modelo lea palabras sueltas
        return " ".join([x.strip() for x in ing_str.split("|") if x.strip()])
    return ""

df["Features"] = df["Ingredients"].apply(join_ingredients)

# =========================
# 2. CREAR EMBEDDINGS
# =========================
print(f"Generando embeddings para {len(df)} recetas...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    df["Features"].tolist(),
    show_progress_bar=True
)

# =========================
# 3. ENTRENAR BUSCADOR VECINAL
# =========================
# Ahora 'embeddings' no estará vacío y funcionará perfectamente
nn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

nn.fit(embeddings)

# =========================
# 4. FUNCIÓN FINAL TOP-5
# =========================
def diet_ia_recommender(ingredientes_usuario, top_k=5):
    emb = embedder.encode([ingredientes_usuario])
    distances, indices = nn.kneighbors(emb, n_neighbors=top_k)

    resultados = []
    for rank, idx in enumerate(indices[0]):
        receta = df.iloc[idx]
        resultados.append({
            "Rank": rank + 1,
            "Similitud": f"{round(float(1 - distances[0][rank]) * 100, 2)}%",
            "Receta": receta["Title"],
            "Ingredientes": receta["Ingredients"],
            "Instrucciones": receta["Instructions"]
        })

    return resultados

Generando embeddings para 17730 recetas...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/555 [00:00<?, ?it/s]

In [29]:
diet_ia_recommender("paprika, lemon, octopus")

[{'Rank': 1,
  'Similitud': 0.5464022159576416,
  'Receta': 'Spanish Octopus ',
  'Ingredientes': '3 (1- to 2-pound each) octopuses, cleaned | 1/2 cup pickling spice | 1/4 cup kosher salt, plus additional for seasoning | 1 tablespoon red pepper flakes | 3/4 cup fresh lemon juice (about 4 lemons)',
  'Instrucciones': 'Cut tentacles from octopus heads (just below eyes) and discard heads. In 5-quart heavy pot, combine tentacles and enough water to cover by two inches. Add pickling spice, salt, pepper flakes, and 1/2 cup lemon juice, and bring to boil. Reduce heat and simmer, covered, until octopus is tender, about 40 minutes. Remove from heat, let rest 5 minutes, then drain. Refrigerate tentacles until chilled, then cut into 3-inch pieces. In large bowl, toss with remaining lemon juice and kosher salt to taste.'},
 {'Rank': 2,
  'Similitud': 0.4941245913505554,
  'Receta': 'Octopus Salad ',
  'Ingredientes': '2 pound frozen octopus, thawed and rinsed | 1/3 cup chopped flat-leaf parsley | 

In [33]:
# Suppress harmless "UNEXPECTED" model loading warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# ==========================================
# 1. DATA PREPARATION
# ==========================================
# Load dataset
data = pd.read_csv('/content/drive/MyDrive/DIET-IA/dataset/recetas.csv')
df = data.copy()

# Rename columns for clarity and consistency
df = df.rename(columns={
    "title": "Title",
    "ingredients": "Ingredients",
    "directions": "Instructions",
    "categories": "Category",
    "calories": "Calories",
    "fat": "Fat",
    "protein": "Protein"
})

# 1. Drop rows with missing values in essential fields
df = df.dropna(subset=["Title", "Ingredients", "Instructions"])

# 2. Remove duplicates by Title and reset index
df = df.drop_duplicates(subset=["Title"]).reset_index(drop=True)

# 3. Process ingredients into a single string for the model
def join_ingredients(ing_str):
    if isinstance(ing_str, str):
        # Convert "item1 | item2" to "item1 item2"
        return " ".join([x.strip() for x in ing_str.split("|") if x.strip()])
    return ""

# CREATE the 'Features' column (Missing in your previous snippet)
df["Features"] = df["Ingredients"].apply(join_ingredients)

# 4. Convert macronutrients to numeric values (coerce errors to NaN)
for col in ["Calories", "Fat", "Protein"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Total recipes processed: {len(df)}")

# ==========================================
# 2. CREATE EMBEDDINGS
# ==========================================
print("Generating text embeddings... this might take a moment.")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    df["Features"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

# ==========================================
# 3. TRAIN NEAREST NEIGHBORS MODEL
# ==========================================
# Initialize the search model using Cosine Similarity
nn = NearestNeighbors(metric="cosine")
nn.fit(embeddings)

# ==========================================
# 4. FINAL RECOMMENDATION FUNCTION (TOP-K + MACROS)
# ==========================================
def diet_ia_recommender(
    user_ingredients,
    top_k=5,
    candidate_pool=50,
    mode=None,              # None (Balanced) | "low_cal" | "low_fat" | "high_protein"
    max_calories=None,
    max_fat=None,
    min_protein=None,
    category=None
):
    # --- 1) Identify candidates based on text similarity ---
    emb = embedder.encode([user_ingredients], convert_to_numpy=True)
    distances, indices = nn.kneighbors(emb, n_neighbors=candidate_pool)

    candidates = df.iloc[indices[0]].copy()
    candidates["Sim"] = 1 - distances[0]

    # --- 2) Apply Category filtering ---
    if category is not None and "Category" in candidates.columns:
        candidates = candidates[
            candidates["Category"].astype(str).str.contains(category, case=False, na=False)
        ]

    # --- 3) Apply Nutritional hard filters ---
    if max_calories is not None and "Calories" in candidates.columns:
        candidates = candidates[candidates["Calories"] <= max_calories]

    if max_fat is not None and "Fat" in candidates.columns:
        candidates = candidates[candidates["Fat"] <= max_fat]

    if min_protein is not None and "Protein" in candidates.columns:
        candidates = candidates[candidates["Protein"] >= min_protein]

    # If filters are too restrictive, revert to original candidate pool
    if len(candidates) == 0:
        candidates = df.iloc[indices[0]].copy()
        candidates["Sim"] = 1 - distances[0]

    # --- 4) Re-rank candidates based on nutritional modes ---
    def minmax_scale(series):
        series = pd.to_numeric(series, errors="coerce").fillna(series.median())
        return (series - series.min()) / (series.max() - series.min() + 1e-9)

    # If macros are missing, default score is pure similarity
    if not all(col in candidates.columns for col in ["Calories", "Fat", "Protein"]):
        candidates["Score"] = candidates["Sim"]
    else:
        cal_n = minmax_scale(candidates["Calories"])
        fat_n = minmax_scale(candidates["Fat"])
        prot_n = minmax_scale(candidates["Protein"])

        if mode == "low_cal":
            candidates["Score"] = 0.75 * candidates["Sim"] + 0.25 * (1 - cal_n)
        elif mode == "low_fat":
            candidates["Score"] = 0.75 * candidates["Sim"] + 0.25 * (1 - fat_n)
        elif mode == "high_protein":
            candidates["Score"] = 0.75 * candidates["Sim"] + 0.25 * prot_n
        else:
            # Balanced Scoring
            candidates["Score"] = (
                0.65 * candidates["Sim"]
                + 0.12 * (1 - cal_n)
                + 0.12 * (1 - fat_n)
                + 0.11 * prot_n
            )

    # Sort by the final calculated Score and take Top-K
    candidates = candidates.sort_values("Score", ascending=False).head(top_k)

    # --- 5) Format output ---
    results = []
    for rank, row in enumerate(candidates.itertuples(), start=1):
        results.append({
            "Rank": rank,
            "Score": round(float(row.Score) * 100, 2),
            "Similarity": round(float(row.Sim) * 100, 2),
            "Recipe": row.Title,
            "Category": getattr(row, "Category", None),
            "Calories": getattr(row, "Calories", None),
            "Fat": getattr(row, "Fat", None),
            "Protein": getattr(row, "Protein", None),
            "Ingredients": row.Ingredients,
            "Instructions": row.Instructions
        })

    return results

Total recipes processed: 17730
Generating text embeddings... this might take a moment.


The following layers were not sharded: encoder.layer.*.intermediate.dense.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.weight, embeddings.position_embeddings.weight, pooler.dense.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.output.dense.weight, embeddings.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, pooler.dense.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.self.key.weight


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/555 [00:00<?, ?it/s]

In [36]:
diet_ia_recommender("rice chicken potato garlic, mode=low_fat")

[{'Rank': 1,
  'Score': 60.67,
  'Similarity': 50.36,
  'Recipe': 'Garlic-Rosemary Turkey with Mushroom Gravy ',
  'Category': 'Garlic | Mushroom | turkey | Roast | Thanksgiving | Rosemary | Fall | Bon Appétit',
  'Calories': 887.0,
  'Fat': 26.0,
  'Protein': 131.0,
  'Ingredients': '6 large heads of garlic, unpeeled, top 1/3 inch cut off to expose garlic | 6 tablespoons olive oil | 1/4 cup (1/2 stick) unsalted butter, room temperature | 2 tablespoons chopped fresh rosemary | 2 tablespoons Dijon mustard | 1/2 teaspoon salt | 1/2 teaspoon ground black pepper | 1 18- to 19-pound turkey; neck, gizzard and heart reserved | 1 large onion, chopped | 2 large celery stalks, chopped | 1 large carrot, chopped | 2 1/2 cups dry white wine | 7 cups (about) canned low-salt chicken broth | 2 pounds mushrooms, thickly sliced | 1/3 cup all purpose flour',
  'Instructions': 'Preheat oven to 350°F. Arrange garlic, cut side up, in 8x8x2-inch glass baking dish. Drizzle with 6 tablespoons oil. Cover dish w

# Guardar Modelo

In [ ]:
import joblib
import os

# Carpeta en Drive
save_path = '/content/drive/MyDrive/DIET-IA/models/Salmeron/'
os.makedirs(save_path, exist_ok=True)

# Guardar el DataFrame
df.to_csv(os.path.join(save_path, 'df_recetas.csv'), index=False)

# Guardar el modelo de embeddings (SentenceTransformer)
embedder.save(os.path.join(save_path, 'embedder'))

# Guardar el NearestNeighbors
joblib.dump(nn, os.path.join(save_path, 'nn_model.pkl'))

print("Modelo, embeddings y datos guardados correctamente en Drive.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo, embeddings y datos guardados correctamente en Drive.


# Cargar Modelo

In [ ]:
import joblib
from sentence_transformers import SentenceTransformer
import pandas as pd

load_path = '/content/drive/MyDrive/DIET-IA/models/Salmeron/'

# 1. Cargar el DataFrame
df = pd.read_csv(load_path + 'df_recetas.csv')

# 2. Cargar el embedder
embedder = SentenceTransformer(load_path + 'embedder')

# 3. Cargar el NearestNeighbors
nn = joblib.load(load_path + 'nn_model.pkl')

print("Modelo, embeddings y datos cargados correctamente desde Drive.")
